In [2]:
import numpy as np
import xarray as xr
import pandas as pd
import os

out = "../Data/Raw/ERA5/era5_geopotential_alps.nc"
zg = xr.open_dataset(out)
z_era5_m = zg["z"].squeeze() / 9.80665
df = pd.read_csv("../Data/Processed/stage1_hugonnet_rgi6_merged.csv")
print("Loaded", len(df), "glaciers")
print("Columns:", df.columns.tolist())

Loaded 3927 glaciers
Columns: ['rgiid', 'period', 'area', 'dmdtda', 'err_dmdtda', 'reg', 'is_cor', 'RGIId', 'Area', 'Zmin', 'Zmax', 'Zmed', 'Slope', 'Aspect', 'Lmax', 'CenLon', 'CenLat', 'northness', 'eastness']


In [4]:
import xarray as xr
import pandas as pd
import numpy as np
import os
lon = xr.DataArray(df["CenLon"].values, dims="g")
lat = xr.DataArray(df["CenLat"].values, dims="g")

# ERA5 grid elevation at each glacier
df["z_era5"] = z_era5_m.sel(longitude=lon, latitude=lat, method="nearest").values

# lapse-rate corrected summer temperature
df["t_jja_corr"] = df["t_jja"] - 0.0065 * (df["Zmed"] - df["z_era5"])

print(df[["Zmed", "z_era5", "t_jja", "t_jja_corr"]].describe())

df.to_csv("../Data/Processed/stage2_model_ready.csv", index=False)
print("\nSaved model-ready table:", len(df), "glaciers")

              Zmed       z_era5        t_jja   t_jja_corr
count  3927.000000  3927.000000  3927.000000  3927.000000
mean   2959.442322  2008.671509    10.789203     4.609192
std     296.430247   309.207672     2.007846     1.823914
min    1716.000000     2.812833     8.086038    -4.701589
25%    2773.000000  1831.887695     9.570605     3.605454
50%    2949.000000  2084.895508    10.573565     4.651069
75%    3132.000000  2194.992188    11.506474     5.643891
max    4451.000000  2425.701416    24.193636    14.337225

Saved model-ready table: 3927 glaciers


In [3]:
import xarray as xr
import pandas as pd
import numpy as np
import os

ERA5_DIR = r"C:\DATA\Dissertation\Glacier_Mass\Glacier_Mass_Balance\Glacier_Mass_Balance\Data\Raw\ERA5\extracted"

# 1. Loading the two monthly files
ds_t = xr.open_dataset(os.path.join(ERA5_DIR, "data_stream-moda_stepType-avgua.nc"))
ds_a = xr.open_dataset(os.path.join(ERA5_DIR, "data_stream-moda_stepType-avgad.nc"))

# 2. Fixing the timestamp misalignment
for d in (ds_t, ds_a):
    d["valid_time"] = d["valid_time"].values.astype("datetime64[M]").astype("datetime64[ns]")

# 3. Unit conversions
t2m_c = ds_t["t2m"] - 273.15                       # Kelvin -> Celsius
days  = ds_a["valid_time"].dt.days_in_month
tp_mm = ds_a["tp"] * 1000 * days                   # m/day -> mm per month
ssrd  = ds_a["ssrd"]

# 4. Restricting to the Hugonnet window (2000-2019)
per = slice("2000-01-01", "2019-12-31")
t2m_c, tp_mm, ssrd = t2m_c.sel(valid_time=per), tp_mm.sel(valid_time=per), ssrd.sel(valid_time=per)

# 5. Climate summaries on the grid
jja      = t2m_c["valid_time"].dt.month.isin([6, 7, 8])
t_jja    = t2m_c.sel(valid_time=jja).mean("valid_time")
p_annual = tp_mm.groupby("valid_time.year").sum().mean("year")
srad_jja = ssrd.sel(valid_time=ssrd["valid_time"].dt.month.isin([6, 7, 8])).mean("valid_time")

# 6. Geopotential -> ERA5 cell elevation
zg = xr.open_dataset("../Data/Raw/ERA5/era5_geopotential_alps.nc")
z_era5_m = zg["z"].squeeze() / 9.80665

# 7. Extract everything at the 3,927 glacier locations
df = pd.read_csv("../Data/Processed/stage1_hugonnet_rgi6_merged.csv")

lon = xr.DataArray(df["CenLon"].values, dims="g")
lat = xr.DataArray(df["CenLat"].values, dims="g")

df["t_jja"]    = t_jja.sel(longitude=lon, latitude=lat, method="nearest").values
df["p_annual"] = p_annual.sel(longitude=lon, latitude=lat, method="nearest").values
df["srad_jja"] = srad_jja.sel(longitude=lon, latitude=lat, method="nearest").values
df["z_era5"]   = z_era5_m.sel(longitude=lon, latitude=lat, method="nearest").values

# 8. Lapse-rate correction
df["t_jja_corr"] = df["t_jja"] - 0.0065 * (df["Zmed"] - df["z_era5"])

print(df[["Zmed", "z_era5", "t_jja", "t_jja_corr", "p_annual"]].describe())
print("\nMissing values:\n", df[["t_jja", "t_jja_corr", "p_annual", "srad_jja"]].isna().sum())

df.to_csv("../Data/Processed/stage2_model_ready.csv", index=False)
print("\nSaved:", len(df), "glaciers with climate features")

              Zmed       z_era5        t_jja   t_jja_corr     p_annual
count  3927.000000  3927.000000  3927.000000  3927.000000  3927.000000
mean   2959.442322  2008.671509    10.789203     4.609192  1567.703170
std     296.430247   309.207672     2.007846     1.823914   288.123875
min    1716.000000     2.812833     8.086038    -4.701589   710.484362
25%    2773.000000  1831.887695     9.570605     3.605454  1366.350174
50%    2949.000000  2084.895508    10.573565     4.651069  1543.352032
75%    3132.000000  2194.992188    11.506474     5.643891  1821.775389
max    4451.000000  2425.701416    24.193636    14.337225  2253.518343

Missing values:
 t_jja         0
t_jja_corr    0
p_annual      0
srad_jja      0
dtype: int64

Saved: 3927 glaciers with climate features


In [2]:
%pip install cdsapi
%pip install os

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement os (from versions: none)
ERROR: No matching distribution found for os


In [1]:
import cdsapi
import os

os.makedirs("../Data/Raw/ERA5", exist_ok=True)
c = cdsapi.Client()

for year in range(2000, 2024):
    out = "../Data/Raw/ERA5/era5_daily_t2m_{year}.nc".format(year=year)
    if os.path.exists(out):
        print("Already have", year)
        continue
        print("Requesting", year, "...")

c.retrieve(
        "derived-era5-single-levels-daily-statistics",
        {
            "product_type": "reanalysis",
            "variable": ["2m_temperature"],
            "year": str(year),                              # ← ONE year only
            "month": [f"{m:02d}" for m in range(1, 13)],
            "day":   [f"{d:02d}" for d in range(1, 32)],
            "daily_statistic": "daily_mean",
            "time_zone": "utc+00:00",
            "frequency": "1_hourly",
            "area": [48, 5, 44, 18],
            "data_format": "netcdf",
            "download_format": "unarchived",
        },
        out,
    )
print("  saved", out)

print("\nAll years done.")

2026-07-15 13:30:18,062 WARNING [2026-06-08T00:00:00] An issue with the following parameters has been identified:

- `maximum_2m_temperature_since_previous_post_processing`
- `minimum_2m_temperature_since_previous_post_processing`
- `10m_wind_gust_since_previous_post_processing`
- `maximum_total_precipitation_rate_since_previous_post_processing`
- `minimum_total_precipitation_rate_since_previous_post_processing`

Please refer to the documented [Known issue](https://confluence.ecmwf.int/display/CKB/ERA5+family+post-processed+daily+statistics+documentation#ERA5familypostprocesseddailystatisticsdocumentation-KnownIssues) for details. **Data downloaded for the parameters listed above should not be used.** A fix is to be released as soon as possible. Please [follow updates on this topic on our Forum announcement](https://forum.ecmwf.int/t/issue-affecting-some-parameters-from-the-era5-post-processed-daily-statistics-on-single-levels/15057).
2026-07-15 13:30:18,064 INFO Request ID is 64e4f838

3d46c0ab7efb4d98896cdbe7de7a186a.nc:   0%|          | 0.00/766k [00:00<?, ?B/s]

  saved ../Data/Raw/ERA5/era5_daily_t2m_2023.nc

All years done.


In [17]:
import xarray as xr

ds = xr.open_dataset("../Data/Raw/ERA5/era5_daily_t2m_2000_2019.nc")
print(ds)

time_dim = "valid_time" if "valid_time" in ds.dims else "time"
print("\nTime steps:", len(ds[time_dim]))
print("From:", ds[time_dim].values[0])
print("To:  ", ds[time_dim].values[-1])

<xarray.Dataset> Size: 1MB
Dimensions:     (valid_time: 365, latitude: 17, longitude: 53)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 3kB 2023-01-01 ... 2023-12-31
  * latitude    (latitude) float64 136B 48.0 47.75 47.5 ... 44.5 44.25 44.0
  * longitude   (longitude) float64 424B 5.0 5.25 5.5 5.75 ... 17.5 17.75 18.0
    number      int64 8B ...
Data variables:
    t2m         (valid_time, latitude, longitude) float32 1MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-07-14T17:17 GRIB to CDM+CF via cfgrib-0.9.1...

Time steps: 365
From: 2023-01-01T00:00:00.000000000
To:   2023-12-31T00:00:00.000000000


In [3]:
import cdsapi
import os

os.makedirs("../Data/Raw/ERA5", exist_ok=True)
c = cdsapi.Client()

out = "../Data/Raw/ERA5/era5_daily_t2m_2000_2023.nc"

c.retrieve(
    "derived-era5-single-levels-daily-statistics",
    {
        "product_type": "reanalysis",
        "variable": ["2m_temperature"],
        "year":  [str(y) for y in range(2000, 2024)],   # 2000-2023
        "month": [f"{m:02d}" for m in range(1, 13)],
        "day":   [f"{d:02d}" for d in range(1, 32)],
        "daily_statistic": "daily_mean",
        "time_zone": "utc+00:00",
        "frequency": "1_hourly",
        "area": [48, 5, 44, 18],
        "data_format": "netcdf",
        "download_format": "unarchived",
    },
    out,
)

HTTPError: 403 Client Error: Forbidden for url: https://cds.climate.copernicus.eu/api/retrieve/v1/processes/derived-era5-single-levels-daily-statistics/execution
cost limits exceeded
Your request is too large, please reduce your selection.

In [8]:
import os

daily_dir = "../Data/Raw/ERA5"
files = sorted([f for f in os.listdir(daily_dir) if f.endswith(".nc")])
print("Number of files:", len(files))
for f in files:
    size_kb = os.path.getsize(os.path.join(daily_dir, f)) / 1e3
    print(f, f"{size_kb:.0f} KB")

Number of files: 4
era5_alps_monthly_2000_2023.nc 1418 KB
era5_daily_t2m_2000_2019.nc 784 KB
era5_daily_t2m_2023.nc 784 KB
era5_geopotential_alps.nc 29 KB


In [9]:
import os

base = "../Data/Raw/ERA5"
for f in ["era5_daily_t2m_2000_2019.nc", "era5_daily_t2m_2023.nc"]:
    p = os.path.join(base, f)
    if os.path.exists(p):
        os.remove(p)
        print("Deleted", f)

Deleted era5_daily_t2m_2000_2019.nc
Deleted era5_daily_t2m_2023.nc


In [2]:
import cdsapi
import os

os.makedirs("../Data/Raw/ERA5/daily", exist_ok=True)
c = cdsapi.Client()

for year in range(2000, 2025):
    out = f"../Data/Raw/ERA5/daily/era5_daily_t2m_{year}.nc"

    if os.path.exists(out):
        print("Already have", year)
        continue

    print("Requesting", year, "...")
    c.retrieve(
        "derived-era5-single-levels-daily-statistics",
        {
            "product_type": "reanalysis",
            "variable": ["2m_temperature"],
            "year": str(year),
            "month": [f"{m:02d}" for m in range(1, 13)],
            "day":   [f"{d:02d}" for d in range(1, 32)],
            "daily_statistic": "daily_mean",
            "time_zone": "utc+00:00",
            "frequency": "1_hourly",
            "area": [48, 5, 44, 18],
            "data_format": "netcdf",
            "download_format": "unarchived",
        },
        out,
    )
    print("  saved", out)

print("\nAll years done.")

Already have 2000
Already have 2001
Already have 2002
Already have 2003
Already have 2004
Already have 2005
Already have 2006
Already have 2007
Already have 2008
Already have 2009
Already have 2010
Already have 2011
Already have 2012
Already have 2013
Already have 2014
Already have 2015
Already have 2016
Already have 2017
Already have 2018
Already have 2019
Already have 2020
Already have 2021
Already have 2022
Already have 2023
Requesting 2024 ...


2026-07-27 13:44:01,752 WARNING [2026-06-08T00:00:00] An issue with the following parameters has been identified:

- `maximum_2m_temperature_since_previous_post_processing`
- `minimum_2m_temperature_since_previous_post_processing`
- `10m_wind_gust_since_previous_post_processing`
- `maximum_total_precipitation_rate_since_previous_post_processing`
- `minimum_total_precipitation_rate_since_previous_post_processing`

Please refer to the documented [Known issue](https://confluence.ecmwf.int/display/CKB/ERA5+family+post-processed+daily+statistics+documentation#ERA5familypostprocesseddailystatisticsdocumentation-KnownIssues) for details. **Data downloaded for the parameters listed above should not be used.** A fix is to be released as soon as possible. Please [follow updates on this topic on our Forum announcement](https://forum.ecmwf.int/t/issue-affecting-some-parameters-from-the-era5-post-processed-daily-statistics-on-single-levels/15057).
2026-07-27 13:44:01,752 INFO Request ID is cdf0195b

d4f4ebeebe073f28a46895d500f6c6e0.nc:   0%|          | 0.00/767k [00:00<?, ?B/s]

  saved ../Data/Raw/ERA5/daily/era5_daily_t2m_2024.nc

All years done.
